# EVA Symbolic — DataSphere (45 GB GPU)
Символьная модель, count-based affinity, большой датасет.


In [ ]:
# 1. Клонирование + установка
import os
if not os.path.exists('/home/jupyter/EVA'):
    !git clone https://github.com/BlackCatSpb/FCF.git /home/jupyter/EVA
%cd /home/jupyter/EVA
!git pull 2>/dev/null; true
!pip install loguru tokenizers numpy faiss-cpu datasets huggingface_hub -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    device = 'cpu'
    print("GPU not found!")
!df -h /home/jupyter

In [ ]:
# 2. Скачивание большого датасета (Wikipedia RU + Gazeta + ConceptNet)
import sys, os, re, time
sys.path.insert(0, '/home/jupyter/EVA')
from eva.symbolic import CharacterVocab
import numpy as np
from datasets import load_dataset

vocab = CharacterVocab()
output_path = '/home/jupyter/EVA/real_data/large_corpus.npy'
os.makedirs('/home/jupyter/EVA/real_data', exist_ok=True)

all_ids = []
total_lines = 0
start_time = time.time()

# === SOURCE 1: Wikipedia RU (streaming) ===
print("[1/2] Wikipedia RU (streaming)...")
try:
    wiki = load_dataset("wikimedia/wikipedia", "20231101.ru", split="train", streaming=True)
    wiki_count = 0
    for item in wiki:
        text = item.get("text", "")
        text = re.sub(r'[^а-яА-ЯёЁ\s.,;:!?\-—«»()""]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) < 20:
            continue
        ids = vocab.encode(text)
        ids.append(0)
        all_ids.extend(ids)
        wiki_count += 1
        total_lines += 1
        if wiki_count % 50000 == 0:
            elapsed = time.time() - start_time
            print(f"  Wiki: {wiki_count:,} articles, {len(all_ids)/1e6:.1f}M tokens, {len(all_ids)*4/1024/1024:.0f} MB")
        if len(all_ids) > 200_000_000:
            break
except Exception as e:
    print(f"  Wiki error: {e}")

# === SOURCE 2: Gazeta (news) ===
print("\n[2/2] Gazeta RU (streaming)...")
try:
    gazeta = load_dataset("IlyaGusev/gazeta", split="train", streaming=True)
    gazeta_count = 0
    for item in gazeta:
        text = item.get("text", "")
        text = re.sub(r'[^а-яА-ЯёЁ\s.,;:!?\-—«»()""]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) < 20:
            continue
        ids = vocab.encode(text)
        ids.append(0)
        all_ids.extend(ids)
        gazeta_count += 1
        total_lines += 1
        if gazeta_count % 10000 == 0:
            elapsed = time.time() - start_time
            print(f"  Gazeta: {gazeta_count:,} articles, {len(all_ids)/1e6:.1f}M tokens, {len(all_ids)*4/1024/1024:.0f} MB")
except Exception as e:
    print(f"  Gazeta error: {e}")

# Save
arr = np.array(all_ids, dtype=np.int32)
np.save(output_path, arr)

elapsed = time.time() - start_time
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"\nDONE: {total_lines:,} lines, {len(all_ids)/1e6:.1f}M tokens, {size_mb:.0f} MB, {elapsed:.0f}s")
!df -h /home/jupyter

In [ ]:
# 3. Обучение — count-based, до сходимости
import sys, os, time, torch, numpy as np, gc
sys.path.insert(0, '/home/jupyter/EVA')
from eva.symbolic import *
from eva.primordial_layer import PrimordialLayer
from eva.config import FCFConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

config = FCFConfig()
config.d_model = 256
config.vocab_size = 156
config.num_heads = 8
config.max_seq_len = 256

layer = PrimordialLayer(config)
if device == 'cuda':
    layer = layer.cuda()

char_vocab = CharacterVocab()
trainer = PotentialTrainer(layer=layer, char_vocab=char_vocab, embed_dim=256,
                           checkpoint_dir='/home/jupyter/EVA/checkpoints/symbolic')

# Dataset
npy_file = '/home/jupyter/EVA/real_data/large_corpus.npy'
if not os.path.exists(npy_file):
    npy_file = '/home/jupyter/EVA/real_data/full_corpus_ids.npy'

all_ids = np.load(npy_file, mmap_mode='r').astype(np.int32)
total_tokens = len(all_ids)
print(f"Dataset: {total_tokens/1e6:.1f}M tokens")

# === TRAINING LOOP ===
BATCH = 128
BLOCK = 128
LOG_INTERVAL = 5000
SAVE_INTERVAL = 50000
MIN_STEPS = 50000
CONV_WINDOW = 50000
AFF_THRESH = 0.0001

pos = 0
start_time = time.time()
prev_pot = 0.5
prev_digrams = 0
steps_no_improvement = 0

print("\nTraining to convergence...\n")

while True:
    if pos + BLOCK + 2 > total_tokens:
        pos = 0
    
    ids_batch, lens = [], []
    for _ in range(BATCH):
        if pos + BLOCK + 2 > total_tokens:
            pos = 0
        end = min(pos + BLOCK, total_tokens)
        chunk = all_ids[pos:end]
        sep = np.where((chunk == 0) | (chunk == 3))[0]
        if len(sep) > 0 and sep[0] < BLOCK // 2:
            end = pos + sep[0] + 1
            chunk = all_ids[pos:end]
        ids = [int(x) for x in chunk if x >= 0][:BLOCK]
        ids_batch.append(ids)
        lens.append(len(ids))
        pos += max(len(ids), 32)
    
    ml = max(lens)
    bt = torch.full((BATCH, ml), 0, dtype=torch.long, device=device)
    for i, ids in enumerate(ids_batch):
        bt[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)
    
    with torch.no_grad():
        layer.eval()
        x = layer.embed(bt)
        layer.forward_transformer(x)
        attn = layer.transformer.attention.last_attention
    
    for i in range(BATCH):
        L = min(lens[i], ml)
        if L < 4:
            trainer.step += 1
            continue
        ids = ids_batch[i][:L]
        am = attn[i].mean(dim=0).cpu().numpy()[:L, :L] if attn is not None else np.eye(L)
        state = trainer.build_assembly(ids, am)
        
        if state.coherence_score > 0.5:
            if attn is not None:
                trainer.potential_field.strengthen_batch(
                    bt[i:i+1, :L], attn[i:i+1, :, :L, :L],
                    confidence=state.coherence_score,
                )
            trainer.valid_assemblies += 1
        else:
            trainer.invalid_assemblies += 1
        
        trainer.total_assemblies += 1
        trainer.step += 1
    
    if trainer.step % 5000 == 0 and trainer.step > 0:
        trainer.grammar.discover_digrams(min_affinity=float(trainer.potential_field.affinity.mean()) + 0.01)
        torch.cuda.empty_cache()
        gc.collect()
    
    if trainer.step % LOG_INTERVAL == 0 and trainer.step > 0:
        elapsed = time.time() - start_time
        lps = trainer.step / max(elapsed, 0.01)
        avg_pot = float(trainer.potential_field.affinity.mean())
        digrams = sum(len(pats) for pats in trainer.grammar.patterns.values())
        
        dpot = avg_pot - prev_pot
        dd = digrams - prev_digrams
        
        print(f"step={trainer.step} | lps={lps:.0f}/s | pot={avg_pot:.4f} (+{dpot:.6f}) | digrams={digrams} (+{dd}) | valid={trainer.valid_assemblies}/{trainer.total_assemblies}")
        
        if trainer.step >= MIN_STEPS and dd < 50:
            steps_no_improvement += LOG_INTERVAL
        else:
            steps_no_improvement = 0
        
        if steps_no_improvement >= CONV_WINDOW:
            print(f"\nCONVERGED! {steps_no_improvement} steps")
            break
        
        prev_pot = avg_pot
        prev_digrams = digrams
    
    if trainer.step % SAVE_INTERVAL == 0:
        trainer.save()

trainer.save(final=True)
elapsed = time.time() - start_time
print(f"\nDone: {trainer.step} steps in {elapsed:.0f}s")
print(f"Pot: {float(trainer.potential_field.affinity.mean()):.4f}")
print(f"Std: {float(trainer.potential_field.affinity.std()):.4f}")
print(f"Digrams: {sum(len(pats) for pats in trainer.grammar.patterns.values())}")

In [ ]:
# 4. Тест генерации
import sys, torch, numpy as np
sys.path.insert(0, '/home/jupyter/EVA')
from eva.symbolic import *

pf = PotentialField(156, 256)
pf.load_state_dict(torch.load('/home/jupyter/EVA/checkpoints/symbolic/final/potential_field.pt', map_location='cpu'))
aff = pf.affinity.cpu().numpy()
vocab = CharacterVocab()

print(f"Pot: mean={aff.mean():.4f}, std={aff.std():.4f}")

# Top digrams
print("\nTOP 20 DIGRAMS:")
pairs = [(float(aff[i,j]), i, j) for i in range(156) for j in range(156) if i != j and aff[i,j] > 0.55]
pairs.sort(reverse=True)
for s, i, j in pairs[:20]:
    print(f"  {vocab.idx_to_char(i)}{vocab.idx_to_char(j)}: {s:.4f}")

# Character order
print("\nCHAR ORDER:")
for word in ['privet', 'chelovek', 'priroda', 'matematika']:
    ids = vocab.encode(word)[1:-1]
    ranks = []
    for k in range(len(ids)-1):
        cont = aff[ids[k]]
        rank = int(np.sum(cont > cont[ids[k+1]]))
        ranks.append(rank)
    print(f"  {word}: avg rank {np.mean(ranks):.0f}/156")

# Word completion
print("\nWORD COMPLETION:")
for prefix in ['mam','pap','knig','chel','zem','vod','ruk','dom','stol']:
    ids = vocab.encode(prefix)[1:-1]
    if not ids: continue
    cont = aff[ids[-1]]
    top5 = np.argsort(cont)[-5:][::-1]
    chars = [vocab.idx_to_char(int(i)) for i in top5]
    print(f"  {prefix}... -> {chars}")

print("\nDone")
!df -h /home/jupyter